# Statistical Consistency Verification

## Overview
This notebook documents p-value discrepancies > 0.05 found during the audit of public A/B test summaries. 
It provides justifications for each discrepancy in accordance with Constitution Principle VI (Transparency & Justification).

### Reference
Analysis methodology follows best practices for statistical consistency verification as discussed in:
- Gelman, A., & Loken, E. (2014). The garden of forking paths. *Working Paper*.
- (1905.08726) https://arxiv.org/abs/1905.08726


In [ ]:
import json
import pandas as pd
from pathlib import Path

# Configuration
PROJECT_ROOT = Path(".")
AUDIT_REPORT_PATH = PROJECT_ROOT / "output" / "audit_report.json"
DISCREPANCY_THRESHOLD = 0.05

def load_audit_records(path: Path) -> list:
    """Load audit records from JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Audit report not found at {path}. Run the pipeline first.")
    with open(path, 'r') as f:
        data = json.load(f)
    return data.get("records", []) if isinstance(data, dict) else data


In [ ]:
def identify_discrepancies(records: list, threshold: float = 0.05) -> pd.DataFrame:
    """
    Identify records where the absolute difference between reported and reconstructed p-values exceeds the threshold.
    
    Args:
        records: List of audit records from the pipeline.
        threshold: Minimum absolute difference to flag as a discrepancy.
    
    Returns:
        DataFrame of flagged records with discrepancy details.
    """
    discrepancies = []
    
    for record in records:
        reported_p = record.get("reported_p_value")
        reconstructed_p = record.get("reconstructed_p_value")
        
        # Skip if values are missing
        if reported_p is None or reconstructed_p is None:
            continue
            
        diff = abs(reported_p - reconstructed_p)
        
        if diff > threshold:
            discrepancies.append({
                "url": record.get("url"),
                "domain": record.get("domain"),
                "reported_p": reported_p,
                "reconstructed_p": reconstructed_p,
                "absolute_diff": diff,
                "test_type": record.get("test_type"),
                "sample_size_mismatch": record.get("data_quality_warning", "").lower().find("sample size") != -1,
                "inconsistent": record.get("is_inconsistent", False)
            })
    
    return pd.DataFrame(discrepancies)


In [ ]:
def generate_justification(record: dict) -> str:
    """
    Generate a justification for the discrepancy based on record metadata.
    Implements Constitution Principle VI: Transparency & Justification.
    """
    reasons = []
    
    # Check for sample size mismatch (common cause of reconstruction error)
    if record.get("sample_size_mismatch"):
        reasons.append("Sample size discrepancy detected between reported and reconstructed data, affecting statistical power and p-value calculation.")
    
    # Check for test type issues
    test_type = record.get("test_type", "unknown")
    if test_type in ["unknown", "mixed"]:
        reasons.append(f"Ambiguous test type ({test_type}) may have led to selection of incorrect reconstruction formula (z-test vs. Fisher's vs. t-test).")
    
    # Check for extreme p-values
    if record["reported_p"] < 0.001 or record["reconstructed_p"] < 0.001:
        reasons.append("Extreme p-values (near 0) are sensitive to rounding errors in reported statistics.")
    
    # Check for effect size issues
    if record.get("inconsistent"):
        reasons.append("Effect size inconsistency (>5% relative difference) suggests potential data reporting error or post-hoc selection bias.")
    
    if not reasons:
        return "Discrepancy exceeds threshold but specific root cause could not be determined from available metadata. May indicate reporting error or unmodeled statistical nuance."
    
    return " | ".join(reasons)


In [ ]:
# Main Execution Block
print("Loading audit records...")
try:
    records = load_audit_records(AUDIT_REPORT_PATH)
    print(f"Loaded {len(records)} records.")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please run the full pipeline (T032) to generate output/audit_report.json before running this notebook.")
    raise SystemExit(1)

print(f"Identifying discrepancies with threshold > {DISCREPANCY_THRESHOLD}...")
df_discrepancies = identify_discrepancies(records, DISCREPANCY_THRESHOLD)

if df_discrepancies.empty:
    print("No discrepancies found exceeding the threshold.")
else:
    print(f"Found {len(df_discrepancies)} discrepancies.")
    
    # Generate justifications
    df_discrepancies["justification"] = df_discrepancies.apply(generate_justification, axis=1)
    
    # Display summary
    print("\n--- Discrepancy Summary ---")
    print(df_discrepancies[["domain", "test_type", "absolute_diff", "sample_size_mismatch"]].describe())
    
    # Save detailed report
    output_path = PROJECT_ROOT / "output" / "discrepancy_justifications.csv"
    df_discrepancies.to_csv(output_path, index=False)
    print(f"\nDetailed justification report saved to: {output_path}")


## Discrepancy Analysis Table
Below are the specific records with p-value discrepancies > 0.05 and their justifications.

In [ ]:
if not df_discrepancies.empty:
    display(df_discrepancies[["url", "domain", "reported_p", "reconstructed_p", "absolute_diff", "justification"]])
else:
    print("No discrepancies to display.")
